# 학습목표
- 한글 텍스트 전처리 방법 실습
  - 토큰화
  - 인코딩
  - 임베딩

- 영화리뷰 데이터 감성분류
  - DNN
  - LSTM
  - BiLSTM
  - Attention
  - Transformer, BERT
  - KoBERT / KoElectra 모델 사용
  - KoBART 모델 사용

- LLM 모델의 종류  

# 텍스트 전처리 기술
  - 학습 말뭉치를 문장으로 세분화
  - 단어 토큰화
  - 어간 추출
  - 표제어 추출(어근 단어로 변환)
  - POS(Part Of Speech) 태깅
  - 정지단어(Stopwords) 식별 및 제거(경우에 따라)
  - 명명된 객체 인식(NER)
    - 문맥을 파악해서 어떤 종류의 객체인지 판별
  - 텍스트 분류
  - 청킹(Chunking, 문장을 유의미한 구문으로 분할)
  - 상호참조 해결(텍스트의 동일한 개체를 가리키는 모든 표현 찾기)

# 한글 텍스트 데이터 전처리 과정
- 한글 텍스트의 특징 영상 : https://www.youtube.com/watch?v=_2MtnLyBdbk
- 토큰화 : 문장 (corpus)에서 요소 (단어, 짧은 문자, 자소, 형태소 등)를 분리하는 작업
    - 오류수정, 결측치 처리
    - 필요한 내용 분리 (한글, 영어 등)
    - 텍스트 증식 (단어 삭제, 단어 추가, 단어 변경, 단어 순서 변경, 의미 통합, 번역후 재번역 등)
    - 토큰화
    - 불용어(Stopword) 처리, 어간 추출, 표제어 추출 등
- 인코딩 : 토큰화된 텍스트를 숫자로 변환하는 작업
    - 빈도수 분석
    - 빈도수에 따라 정렬
    - 정렬된 순서에 따른 인덱스를 부여 (1부터 부여)
    - padding : 단어 집합의 갯수를 동일하게 맞추는 작업
      - 긴 것은 자르고 짧은 0으로 붙임    
- 임베딩 : 인코딩된 데이터에서 텍스트의 특성을 추출하는 작업
    - 방향과 크기로 2차원 데이터로 벡터화하는 작업

# 네이버 영화 리뷰 감성 분류하기

- 한국어 데이터는 토큰화(tokenization)를 할 때 형태소 분석기를 사용

- 총 200,000개 리뷰로 구성된 데이터로 영화 리뷰에 대한 텍스트와 해당 리뷰가 긍정인 경우 1을 부정인 경우 0으로 표시한 레이블로 구성

- 데이터 다운로드 : https://github.com/e9t/nsmc/

- 구글 드라이브 연동

from google.colab import drive
drive.mount('/content/drive')

- 작업폴더로 이동

%cd /content/drive/MyDrive/Colab Notebooks/오르미 1기/텍스트마이닝

In [6]:
import os
from pathlib import Path

# Get the notebook directory and check for data folder
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "notebooks":
    DATA_DIR = NOTEBOOK_DIR.parent / "data"
else:
    DATA_DIR = NOTEBOOK_DIR / "data"

# Check if data directory exists
if DATA_DIR.exists():
    print(f"✓ Data directory found: {DATA_DIR}")
    # List files in data directory
    files = os.listdir(DATA_DIR)
    print(f"Files in data directory: {files}")
    
    # Check for the specific txt files
    required_files = ['ratings_train.txt', 'ratings_test.txt']
    for file in required_files:
        if file in files:
            file_path = DATA_DIR / file
            file_size = file_path.stat().st_size / (1024 * 1024)  # MB
            print(f"✓ {file} found ({file_size:.2f} MB)")
        else:
            print(f"✗ {file} not found")
else:
    print(f"✗ Data directory not found at: {DATA_DIR}")
    print("Please ensure the data files are in the correct location.")

✓ Data directory found: d:\repos\tonylee\goorm\ai-track\daily-pocs\week-03\day-3\data
Files in data directory: ['naver_shopping.txt', 'ratings.txt', 'ratings_test.txt', 'ratings_train.txt', '프로젝트010_네이버영화댓글감성분류_LSTM_BiLSTM_Attension_BERT_배포.ipynb']
✓ ratings_train.txt found (13.95 MB)
✓ ratings_test.txt found (4.67 MB)


- 데이터파일 로드

In [8]:
import pandas as pd

# Load data files using DATA_DIR
train_data = pd.read_table(DATA_DIR / "ratings_train.txt")
test_data = pd.read_table(DATA_DIR / "ratings_test.txt")

# 데이터의 길이 확인
print(f"Training data: {len(train_data):,} samples")
print(f"Test data: {len(test_data):,} samples")

Training data: 150,000 samples
Test data: 50,000 samples


In [9]:
train_data.head(5)

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [10]:
test_data.head(5)

,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,GDNTOPCLASSINTHECLUB,0
2,8544678,뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임... 돈주고 보기에는....,0
4,6723715,3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??,0


- 데이터 확인

In [11]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB


In [12]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        50000 non-null  int64 
 1   document  49997 non-null  object
 2   label     50000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.1+ MB


- 라벨의 편향 유무 확인
  - 긍정, 부정의 갯수가 차이가 많이 나면 많은 쪽으로 편향된 예측이 나옴

In [13]:
train_data['label'].value_counts()

label
0    75173
1    74827
Name: count, dtype: int64

In [14]:
test_data['label'].value_counts()

label
1    25173
0    24827
Name: count, dtype: int64

- 결측치가 있는 데이터 출력

- 결측치 삭제

In [15]:
# how='any' : 한 컬럼이라도 결측치가 있다면 데이터 삭제
train_data = train_data.dropna(how='any')
test_data = test_data.dropna(how='any')

In [16]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 149995 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        149995 non-null  int64 
 1   document  149995 non-null  object
 2   label     149995 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 4.6+ MB


In [17]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 49997 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        49997 non-null  int64 
 1   document  49997 non-null  object
 2   label     49997 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.5+ MB


- 댓글에서 한글하고 공백만 추출

In [18]:
# str : document가 object 타입이어서 문자열 함수인 replace()를 쓰기위해서 문자열로 변환
# replace(A, B) : A를 B로 변경
# [가-힣\s] : 한글과 공백 의미
# ^ : not
# regex=True : 처리하고 값을 저장
train_data["document"] = train_data["document"].str.replace("[^가-힣 ]","", regex=True)
test_data["document"] = test_data["document"].str.replace("[^가-힣 ]","", regex=True)

In [19]:
train_data.head(5)

,id,document,label
0,9976970,아 더빙 진짜 짜증나네요 목소리,0
1,3819312,흠포스터보고 초딩영화줄오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 솔직히 재미는 없다평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화스파이더맨에서 늙어보이기만 했던 커스틴 던...,1


In [20]:
test_data.head(5)

,id,document,label
0,6270596,굳,1
1,9274899,,0
2,8544678,뭐야 이 평점들은 나쁘진 않지만 점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임 돈주고 보기에는,0
4,6723715,만 아니었어도 별 다섯 개 줬을텐데 왜 로 나와서 제 심기를 불편하게 하죠,0


- 한글만 추출해서 생기는 빈 댓글 확인

In [30]:
# 빈 문자열이나 공백만 있는 댓글 확인
print("=== Train Data ===")
print(f"전체 데이터 수: {len(train_data):,}")

# 빈 문자열 확인
empty_train = train_data[train_data['document'] == '']
print(f"완전히 빈 문자열: {len(empty_train):,}")

# 공백만 있는 문자열 확인
whitespace_train = train_data[train_data['document'].str.strip() == '']
print(f"공백만 있는 문자열: {len(whitespace_train):,}")

print(f"\n제거해야 할 총 데이터: {len(whitespace_train):,}")

# 샘플 출력
if len(whitespace_train) > 0:
    print("\n빈 댓글 샘플:")
    print(whitespace_train.head(10))

=== Train Data ===
전체 데이터 수: 148,385
완전히 빈 문자열: 0
공백만 있는 문자열: 0

제거해야 할 총 데이터: 0


- 빈공백만 있는 데이터 삭제

In [31]:
# 공백만 있거나 빈 문자열인 행 제거
print("=== 빈 데이터 제거 전 ===")
print(f"Train data: {len(train_data):,}")
print(f"Test data: {len(test_data):,}")

# strip()으로 공백 제거 후 빈 문자열인 행 제거
train_data = train_data[train_data['document'].str.strip() != '']
test_data = test_data[test_data['document'].str.strip() != '']

print("\n=== 빈 데이터 제거 후 ===")
print(f"Train data: {len(train_data):,}")
print(f"Test data: {len(test_data):,}")

=== 빈 데이터 제거 전 ===
Train data: 148,385
Test data: 49,430

=== 빈 데이터 제거 후 ===
Train data: 148,385
Test data: 49,430


In [32]:
# 인덱스 재설정 (중요: 빈 행 제거 후 인덱스가 불연속적이므로 재설정)
train_data.reset_index(drop=True, inplace=True)
test_data.reset_index(drop=True, inplace=True)

print("인덱스 재설정 완료")
print(f"Train data shape: {train_data.shape}")
print(f"Test data shape: {test_data.shape}")

인덱스 재설정 완료
Train data shape: (148385, 3)
Test data shape: (49430, 3)


In [33]:
# 최종 데이터 확인
print("=== 최종 데이터 정보 ===")
print("\nTrain Data:")
train_data.info()
print("\nTest Data:")
test_data.info()

print("\n샘플 데이터:")
print(train_data.head())

=== 최종 데이터 정보 ===

Train Data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148385 entries, 0 to 148384
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        148385 non-null  int64 
 1   document  148385 non-null  object
 2   label     148385 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB

Test Data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49430 entries, 0 to 49429
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        49430 non-null  int64 
 1   document  49430 non-null  object
 2   label     49430 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.1+ MB

샘플 데이터:
         id                                           document  label
0   9976970                                  아 더빙 진짜 짜증나네요 목소리      0
1   3819312                         흠포스터보고 초딩영화줄오버연기조차 가볍지 않구나      1
2  10265843                        

- 토큰화
  - 한글을 형태소로 토큰화
  - 불용어 (stopword) 삭제

In [26]:
!pip install konlpy

  Using cached konlpy-0.6.0-py2.py3-none-any.whl.metadata (1.9 kB)
Using cached konlpy-0.6.0-py2.py3-none-any.whl (19.4 MB)


In [34]:
import pandas as pd
from konlpy.tag import Okt

# 1. Initialize the analyzer
okt = Okt()

# 2. Use our actual train_data (already loaded and preprocessed)
print("--- Original Reviews from Train Data (First 5 samples) ---")
print(train_data[['document', 'label']].head())
print("\n" + "="*50 + "\n")

# 3. Test on a single sample review from our actual data
sample_review = train_data['document'].iloc[0]
sample_label = train_data['label'].iloc[0]

# Morpheme analysis (basic tokenization)
tokens = okt.morphs(sample_review)
print(f"--- 1. Morphemes (okt.morphs) ---")
print(f"Sample Review: '{sample_review}'")
print(f"Label: {sample_label} ({'긍정' if sample_label == 1 else '부정'})")
print(f"Result: {tokens}")
print(f"Token count: {len(tokens)}")

--- Original Reviews from Train Data (First 5 samples) ---
                                            document  label
0                                  아 더빙 진짜 짜증나네요 목소리      0
1                         흠포스터보고 초딩영화줄오버연기조차 가볍지 않구나      1
2                                  너무재밓었다그래서보는것을추천한다      0
3                          교도소 이야기구먼 솔직히 재미는 없다평점 조정      0
4  사이몬페그의 익살스런 연기가 돋보였던 영화스파이더맨에서 늙어보이기만 했던 커스틴 던...      1


--- 1. Morphemes (okt.morphs) ---
Sample Review: '아 더빙 진짜 짜증나네요 목소리'
Label: 0 (부정)
Result: ['아', '더빙', '진짜', '짜증나네요', '목소리']
Token count: 5


In [35]:
# Extracting Nouns only (often used as features)
nouns = okt.nouns(sample_review)
print(f"\n--- 3. Extracting Nouns (okt.nouns) ---")
print(f"Result: {nouns}")


--- 3. Extracting Nouns (okt.nouns) ---
Result: ['더빙', '진짜', '목소리']


In [37]:
# Define a function to process a single document (review)
def tokenize_and_filter(text):
    """
    Tokenizes the text using Okt and returns a list of nouns.
    """
    # Use okt.nouns() to extract only nouns
    nouns = okt.nouns(text)
    return nouns

# Apply the function to the 'review' column
df['tokenized_review'] = df['review'].apply(tokenize_and_filter)

# Display the results
print(f"\n--- 4. Tokenization Applied to DataFrame ---")
print(df[['review', 'tokenized_review']])


--- 4. Tokenization Applied to DataFrame ---
                           review         tokenized_review
0  이 영화 정말 재미있었어요! 배우들의 연기가 최고예요.  [이, 영화, 정말, 배우, 연기, 최고]
1         별로였어요. 시간이 아까운 느낌이었습니다.                 [별로, 시간]
2   KoNLPy를 사용하여 한국어 리뷰를 분석해 봅시다.     [를, 사용, 한국어, 리뷰, 분석]


- 토큰화된 결과를 저장

In [40]:
# Tokenization using Okt for the actual movie review data
def tokenize_text(text):
	"""토큰화 함수: 명사만 추출하여 공백으로 연결"""
	if pd.isna(text) or text.strip() == '':
		return ''
	nouns = okt.nouns(text)
	return ' '.join(nouns)

# 실제 영화 리뷰 데이터에서 샘플 추출 (처리 시간을 위해 1000개만 사용)
sample_size = 1000
train_sample = train_data.head(sample_size).copy()

print(f"샘플 데이터 크기: {len(train_sample):,}개")
print("토큰화 진행 중...")

# 토큰화 적용
train_sample['tokenized_document'] = train_sample['document'].apply(tokenize_text)

# 빈 토큰화 결과 제거
train_sample = train_sample[train_sample['tokenized_document'].str.strip() != '']

print(f"토큰화 후 데이터 크기: {len(train_sample):,}개")
print("\n토큰화 결과 샘플:")
print(train_sample[['document', 'tokenized_document', 'label']].head())

샘플 데이터 크기: 1,000개
토큰화 진행 중...
토큰화 후 데이터 크기: 970개

토큰화 결과 샘플:
                                            document  \
0                                  아 더빙 진짜 짜증나네요 목소리   
1                         흠포스터보고 초딩영화줄오버연기조차 가볍지 않구나   
2                                  너무재밓었다그래서보는것을추천한다   
3                          교도소 이야기구먼 솔직히 재미는 없다평점 조정   
4  사이몬페그의 익살스런 연기가 돋보였던 영화스파이더맨에서 늙어보이기만 했던 커스틴 던...   

         tokenized_document  label  
0                 더빙 진짜 목소리      0  
1    흠 포스터 보고 초딩 영화 줄 오버 연기      1  
2            무재 밓었 다그 래서 추천      0  
3       교도소 이야기 구먼 재미 평점 조정      0  
4  몬페 의 연기 영화 스파이더맨 커스틴 던스트      1  
토큰화 후 데이터 크기: 970개

토큰화 결과 샘플:
                                            document  \
0                                  아 더빙 진짜 짜증나네요 목소리   
1                         흠포스터보고 초딩영화줄오버연기조차 가볍지 않구나   
2                                  너무재밓었다그래서보는것을추천한다   
3                          교도소 이야기구먼 솔직히 재미는 없다평점 조정   
4  사이몬페그의 익살스런 연기가 돋보였던 영화스파이더맨에서 늙어보이기만 했던 커스틴 던...   

         tok

- 저장된 파일 불러오기

- 라벨 데이터를 pickle로 저장

# 인코딩

- Tokenizer()로 인코딩
  - fit_on_texts() : 빈도수 분석, 정렬, 인덱스 부여
  - texts_to_sequences() : 인코딩
- padding : pad_sequences()

- padding

- 패딩 결과 저장

- 패딩 파일 불러오기

# 학습 모델 설계 (Dense 층으로 구성)

- Embedding() 층을 이용해서 워드 임베딩을 수행
- Dense() 층을 이용해서 학습

- 컴파일

- 예측하기

- 학습된 모델을 이용하여 직접 댓글을 입력해서 긍정/부정 판별하기

# LSTM을 이용한 모델 설계

- 모델 비교하기

# BiLSTM 적용하기

### 다 대 다(many-to-many) 문제를 푸는 경우의 양방향 LSTM
  - 양방향 LSTM은 두 개의 독립적인 LSTM 아키텍처를 함께 사용하는 구조
  - 주황색 LSTM 셀은 순차적으로 입력을 받음 -> 사람처럼 문장을 왼쪽 단어부터 순차적으로 읽음
  - 양방향 LSTM은 뒤의 문맥까지 고려하기 위해서 문장을 오른쪽에서 반대로 읽는 역방향의 LSTM 셀(초록색)을 함께 사용

<center>  
<img src="https://arome1004.cafe24.com/images/deeplearning/bilstm1.png" width=40%>   
</center>  

### 다 대 일(many-to-one) 문제를 푸는 경우의 양방향 LSTM
  - 일반적으로 순방향 LSTM은 마지막 시점의 은닉 상태를 출력층으로 보내서 텍스트 분류를 수행
  - 하지만, 역방향 LSTM은 x<sub>4</sub>만 본 상태이므로 역방향 LSTM이 텍스트 분류를 위한 유용한 정보를 갖고 있다고 기대하기 어려움

<center>  
<img src="https://arome1004.cafe24.com/images/deeplearning/bilstm2.png" width=40%>   
</center>  

- 케라스에서는 양방향 LSTM을 사용하면 <font color=red>return_sequences=False</font>로 설정
- 순방향 LSTM의 경우에는 마지막 시점의 은닉 상태를 반환하고, 역방향 LSTM의 경우에는 첫번째 시점의 은닉 상태를 반환하여 양방향 LSTM으로 텍스트 분류를 수행

<center>  
<img src="https://arome1004.cafe24.com/images/deeplearning/bilstm3.png" width=40%>   
</center>  

- BiLSTM을 적용하여 긍정/부정 판별하기

- 모델 비교하기

# Attention 적용하기

- 어텐션의 기본 아이디어
  - 시간이 지나면 이전시점의 데이터가 다음시점에 미치는 영향이 동일하게 줄어드는가 ?
  - 과거시점의 데이터들이 현재시점에 미치는 영향이 동일비율로 감소하는가 ?


<center>  
<img src="https://arome1004.cafe24.com/images/deeplearning/attension_concept.png" width=40%>   
</center>

- 어텐션 개요
  - 디코더에서 출력 단어를 예측하는 매 시점(time step)마다, 인코더에서의 전체 입력 문장을 다시 한 번 참고
  - 단, 전체 입력 문장을 전부 다 동일한 비율로 참고하는 것이 아니라, 해당 시점에서 예측해야할 단어와 연관이 있는 입력 단어 부분을 좀 더 집중(attention)

<center>  
<img src="https://arome1004.cafe24.com/images/deeplearning/dotproductattention4_final.png" width=40%>   
</center>    


  

- 모델 비교하기

# Transformer (BERT (Bidirectional Encoder Representations from Transformers))

### 시퀀스-투-시퀀스(Sequence-to-Sequence)
  - 번역기에서 대표적으로 사용되는 모델
  - 인코더와 디코더라는 두 개의 모듈로 구성
  - 인코더 :  입력 문장의 모든 단어들을 순차적으로 입력받은 뒤에 마지막에 이 모든 단어 정보들을 압축해서 하나의 벡터로 만듬 (context vector)
  - 디코더 : 컨텍스트 벡터를 받아서 번역된 단어를 한 개씩 순차적으로 출력

<center>  
<img src="https://arome1004.cafe24.com/images/deeplearning/encoder_decoder.png" width=60%>   
</center>    

- RNN에 기반한 seq2seq 모델의 두 가지 문제
  - 하나의 고정된 크기의 벡터에 모든 정보를 압축하려고 하니까 정보 손실이 발생
  - RNN의 고질적인 문제인 기울기 소실(vanishing gradient) 문제가 존재

### Transformer
  - 필요한 것은 주의 집중(Attention is All You Need) 논문으로 2017년 발표
  - 순환과 컨볼루션을 완전히 배제하고 오로지 Attention 메커니즘에만 기반을 둔 새롭고 단순한 신경망 아키텍처

  - 인코더와 디코더로 구성
  - 인코더
    - 트랜스포머는 하이퍼파라미터인 num_layers 개수의 인코더 층을 쌓음
    - 하나의 인코더 층은 크게 셀프 어텐션과 피드 포워드 신경망의 2개의 서브층(sublayer)으로 구성
      - Multi-head Self-Attention : 셀프 어텐션을 병렬적으로 사용
      - Position-wise FFNN : 일반적인 피드 포워드 신경망
    - 인코딩의 출력은 디코더로 전달

  - 디코더
    - num_layers 개수의 디코더 층을 쌓음  
    - 인코더에서 보낸 출력을 디코더 층의 연산에 사용
    - 디코더는 3개 층으로 구성
      - Masked Multi-head Self-Attention (Self Attention과 Look-ahead mask): 현재 시점의 예측에서 현재 시점보다 미래에 있는 단어들을 참고하지 못하도록 룩-어헤드 마스크(look-ahead mask)를 사용
        - 트랜스포머는 문장 행렬로 입력을 한 번에 받으므로 현재 시점의 단어를 예측하고자 할 때, 입력 문장 행렬로부터 미래 시점의 단어까지도 참고할 수 있는 현상이 발생
      - Multi-head Self-Attention (Encoder-Decoder Attention) : 멀티 헤드 어텐션을 수행
      - Position-wise FFNN

<center>  
<img src="https://arome1004.cafe24.com/images/deeplearning/transformer_attention_overview.png" width=60%>   
</center>    




### self attention
  - 참고 : https://ratsgo.github.io/nlpbook/docs/language_model/tr_self_attention/

- self attention은 쿼리(Q), 키(K), 값(V) 3개 요소 사이의 문맥적 관계성을 추출하는 과정
  - W<sub>Q</sub>, W<sub>K</sub>, W<sub>V</sub>는 학습 과정에서 최적값으로 업데이트됨

<pre>
Q = X x W<sub>Q</sub>
K = X x W<sub>K</sub>
V = X x W<sub>V</sub>

Attention(Q, K, V) = softmax(QK<sup>T</sup> / d<sub>K</sub>)V
</pre>

- Multi-head Attention
  - 셀프 어텐션(self attention)을 여러 번 수행한 것
  - 같은 데이터를 두고 독자(헤드) 여러 명이 각자 읽는 형태

<center>  
<img src="https://arome1004.cafe24.com/images/deeplearning/self_attension.png" width=40%>   
</center>    


- 디코더의 Multi-head Attention
  - 인코더의 입력이 한글이고 디코더의 출력이 영어인 경우
  
<center>  
<img src="https://arome1004.cafe24.com/images/deeplearning/self_attension2.png" width=40%>   
</center>    



- Masked Multi-head Attension
  - 학습과정에서는 인코더에 "어제 카페 갔다 거기 사람 많다"가, 디코더에 입력된 상황이라면 트랜스포머 모델은 다음 영어 단어 I를 맞추도록 학습
  - 하지만 학습 과정에서 모델에 이번에 맞춰야할 정답인 I를 알려주게 되면 학습하는 의미가 없어짐
  - 따라서 정답을 포함한 미래 정보를 셀프 어텐션 계산에서 제외하게 하는 것
  - 예측할 대상은 확률은 높이고 다른 단어들의 확률은 낮게 조정
  
<center>  
<img src="https://arome1004.cafe24.com/images/deeplearning/self_attension3.png" width=40%>   
</center>    


### BERT
- BERT 모형은 2018년 11월 구글이 공개한 인공지능(AI) 언어 모델
- BERT 모형의 특징 : 사전학습, 문맥학습, 파인튜닝
  - (사전학습) 위키피디아 같은 아주 큰 데이터들을 사용하여 '언어 이해' 모델을 사전학습(Pre-training)
  - (문맥학습1) 문장 순서를 학습하여 다음에 나온 문장이 순서에 맞는 문장인지 학습
    - 문장1: 저 남자는 회사에 출근했다
    - 문장2: 회사에 출근하자마자 저 남자는 커피를 끓여 마셨다. (순서가 맞음)
    - 문장3: 저 여자는 퇴근하려 한다.
    - 문장4: 강아지는 예쁘다. (순서가 틀린 문장)

  - (문맥학습2) 양방향으로 학습하여 가려진 단어를 맞춘다.
    - 문장 : 저 남자는 (①)에 출근했다. 회사에 출근하자마자 저 남자는 (②)를 마셨다.
    - ① : 회사 ② : 커피

  - (파인튜닝) 생성된 모델을 최적화하여 새로운 과제를 해결


- 학습 목표

<center>  
<img src="https://arome1004.cafe24.com/images/deeplearning/bert.png" width=40%>   
</center>    


- transformer 설치

In [ ]:
# 코랩은 설치되어 있음
!pip install transformers

- 데이터 새로 불러오기

- 결측치 제거

- 한글과 공백만 추출, 댓글이 없는 데이터 삭제

- 형태소 분리 후 다시 연결 - BERT 토큰화 전에 정규화 작업

- 파일로 저장하기

- 저장된 파일 불러오기

- 일부 데이터만 사용

- 훈련-테스트 데이터 분할

- Hugging Face Dataset 객체로 변환

# 3가지 타입의 한국어 언어모델
- Encoder 중심의 모델 : BERT 계열

<table>
<tr align="center">
  <td width="50"><b>모델명</b>
  <td width="50"><b>개발자</b>
  <td width="100"><b>학습데이터</b>
  <td width="100"><b>Tokenizer</b>
  <td width="100"><b>단어수</b>
  <td width="50"><b>파라미터수</b>
<tr align="center">
  <td>KorBERT
  <td>ETRI
  <td>뉴스/백과사전<br>23GB
  <td>Mophologpy,<br>WordPiece
  <td>30,349(Mophologpy),<br>30,797(WordPiece)
  <td>110M
<tr align="center">
  <td>KoBERT
  <td>SKT
  <td>위키피디아<br>20M
  <td>Sentencce-Piece
  <td>8,002
  <td>92M
<tr align="center">
  <td>HanBERT
  <td>투블럭AI
  <td>일반/특허문서<br>70GB
  <td>Moran
  <td>54,000
  <td>128M
<tr align="center">
  <td>KoreALBERT
  <td>삼성SDS
  <td>위키피디아/나무위키/뉴스<br>책 줄거리 요약 등<br>43GB
  <td>Sentencce-Piece
  <td>32,000
  <td>12M, 18M
<tr align="center">
  <td>KLUE-BERT
  <td>Klue project
  <td>모두의말뭉치/CC-100-Kor/<br>나무위키/뉴스/청원 등<br>63GB
  <td>Morpheme-based<br>subword
  <td>32,000
  <td>111M
<tr align="center">
  <td>KRBERT
  <td>서울대
  <td>위키피디아/뉴스
  <td>WordPiece
  <td>16,424(Character)<br>12,367(Subcharacter)
  <td>99M(Character)<br>96M(Subcharacter)
<tr align="center">
  <td>DistillKoBERT
  <td>개인(박장원)
  <td>위키피디아/나무위키/<br>뉴스 등
  <td>Sentence-Piece
  <td>30,522
  <td>27.8M
<tr align="center">
  <td>KcBERT
  <td>개인(이준범)
  <td>네이버뉴스의 댓글/대댓글
  <td>Word-Piece
  <td>30,000
  <td>109M
<tr align="center">
  <td>KcELETRA
  <td>개인(이준범)
  <td>네이버뉴스의 댓글/대댓글
  <td>Word-Piece
  <td>30,000
  <td>124M
<tr align="center">
  <td>KoBigBird
  <td>개인(박장원)
  <td>위키피디아/뉴스/모두의<br>말뭉치/Common Crawl 등
  <td>Word-Piece
  <td>32,500
  <td>113.8M                   
</table>

- Decoder 중심의 모델 : GPT 계열

<table>
<tr align="center">
  <td width="50"><b>모델명</b>
  <td width="50"><b>개발자</b>
  <td width="100"><b>학습데이터</b>
  <td width="100"><b>Tokenizer</b>
  <td width="100"><b>단어수</b>
  <td width="50"><b>파라미터수</b>
<tr align="center">
  <td>KoGPT2
  <td>SKT
  <td>위키피디아, 뉴스, 나무위키,<br>네이버영화리뷰<br>한국어 Common Crawl<br>152M
  <td>Character BPE
  <td>51,200
  <td>125M
<tr align="center">
  <td>KoGPT-Trinity
  <td>SKT
  <td>Ko-DATA dataset<br>1.2B
  <td>-
  <td>51,200
  <td>1.2B
<tr align="center">
  <td>HyperCLOVA
  <td>NAVER
  <td>뉴스/카페/블로그/지식in/<br>웹문서/댓글 등 네이버<br>수집문서, 모두의 말뭉치<br>위키피디아 등<br>562B
  <td>Morpheme-aware byte-<br>level BPE
  <td>-
  <td>82.0B
<tr align="center">
  <td>KoGPT
  <td>Kakaobrain
  <td>200B
  <td>-
  <td>64,512
  <td>6.0B
</table>        

- Encoder-Decoder 모델 : Seq2Seq 계열

<table>
<tr align="center">
  <td width="50"><b>모델명</b>
  <td width="50"><b>개발자</b>
  <td width="100"><b>학습데이터</b>
  <td width="100"><b>Tokenizer</b>
  <td width="100"><b>단어수</b>
  <td width="50"><b>파라미터수</b>
<tr align="center">
  <td>KoBART
  <td>SKT
  <td>위키피디아/뉴스/모두의<br>말뭉치/청와대 국민청원 등<br>0.27B
  <td>Character BPE
  <td>30,000
  <td>124M
<tr align="center">
  <td>KE-T5
  <td>KETI
  <td>한국어와 영어 데이터가<br> 7:3의 비율인 데이터<br>30GB
  <td>Sentence-Piece
  <td>64,000
  <td>247M
<tr align="center">
  <td>ET5
  <td>ETRI
  <td>위키피디아/뉴스/방송<br>대본/영화 드라마 대본 등<br>136GB
  <td>Sentence-Piece
  <td>45,100
  <td>60M
<tr align="center">
  <td>EXAONE
  <td>LG AI연구원
  <td>말뭉치 600B, 이미지,<br>텍스트 Pair 데이터 250M이상
  <td>-
  <td>-
  <td>300B
</table>        

- 대표적인 Transformer 모델

<center>  
<img src="https://arome1004.cafe24.com/images/deeplearning/transformer_model01.png" width=60%>   
</center>    


# KoBERT
- 2019년에 SKT Brain에서 공개
- 한국어 자연어 처리를 위해 최적화된 BERT(Bidirectional Encoder Representations from Transformers) 기반 모델
- KoBERT는 BERT 모델을 한국어 데이터로 사전 학습하여, 한국어의 미묘한 문맥과 의미를 더 잘 파악할 수 있도록 만들어진 라이브러리

- 토큰화 / 인코딩

- 모델 훈련

- 예측

# KoELECTRA
- ELECTRA는 Replaced Token Detection, 즉 generator에서 나온 token을 보고 discriminator에서 "real" token인지 "fake" token인지 판별하는 방법으로 학습
- 이 방법은 모든 input token에 대해 학습할 수 있다는 장점을 가지며, BERT 등과 비교했을 때 더 좋은 성능

- KoELECTRA는 34GB의 한국어 text로 학습하였고, 이를 통해 나온 KoELECTRA-Base와 KoELECTRA-Small 두 가지 모델을 배포
- KoELECTRA는 Wordpiece 사용, 모델 s3 업로드 등을 통해 OS 상관없이 Transformers 라이브러리만 설치하면 곧바로 사용 가능

<center>  
<img src="https://arome1004.cafe24.com/images/deeplearning/koelectra01.png" width=60%>   
</center>

- 토큰화

- 모델 훈련

- 예측

# KoBART
- 2020년 SKT에서 공개한 BART(Bidirectional and Auto-Regressive Transformers)의 한국어 버전
- 입력 텍스트 일부에 노이즈를 추가하여 이를 다시 원문으로 복구하는 autoencoder의 형태로 학습
- Text Infilling 노이즈 함수를 사용하여 40GB 이상의 한국어 텍스트에 대해서 학습한 한국어 <font color=red>encoder-decoder 언어 모델</font>

<center>  
<img src="https://arome1004.cafe24.com/images/deeplearning/bart01.png" width=50%>   
</center>

- 토큰화

- 모델 훈련

- 예측

# LLM 모델의 종류
- 엘모(ELMo)
  - 2018년 출시
  - 앨런NLP(AllenNLP)의 심층 맥락화 단어 표현 LLM
  - 단어 사용의 복잡한 특징, 그리고 언어적 맥락에 따라 그 단어 사용이 어떻게 달라지는지를 모델링
  - 원본 모델은 9,360만 개의 매개변수를 사용하며 1B 워드 벤치마크로 학습

- 버트(BERT)
  - 구글 AI가 자사 트랜스포머 신경망 아키텍처를 기반으로
2018년에 출시한 언어 모델
  - 모든 레이어에서 왼쪽과 오른쪽의 맥락에 대한 공동 조건화를 통해 레이블 없는 텍스트로부터 심층 양방향 표현을 사전 학습하도록 설계
  - 초기에 사용된 두 모델의 크기는 각각 매개변수 1억 개와 3억 4,000만 개
  - 영어 위키피디아와 토론토 북 코퍼스를 사용해 학습  

- T5 (Text-To-Text Transfer Transformer)
  - 구글의 2020년에 출시된 T5 모델은 이전 모델에 적용된 최고의 전이 학습 기법을 기반으로 새로운 모델을 합성
  - 800GB의 영어용 표준 C4를 사전학습 데이터 집합으로 사용
  - T5는 모든 NLP 작업을 통합 텍스트-투-텍스트 형식으로 재구성
  - 클래스 레이블 또는 입력의 범위만 출력할 수 있는 BERT 스타일 모델과 달리, 입력과 출력이 항상 텍스트 문자열
  - 기본 T5 모델에는 총 2억 2,000만 개의 매개변수가 사용

- GPT (Generative Pretrained Transformer)
  - 트랜스포머 신경망 아키텍처를 기반으로 제작
  - 오픈AI가 2018년에 출시한 모델로, 약 1억 1,700만 개의 매개변수를 사용
  - 토론토 북 코퍼스를 사용해 사전 학습된 단방향 트랜스포머이며 인과적 언어 모델링(CLM) 목표에 따라, 시퀀스의 다음 토큰을 예측하도록 학습
  - GPT-2(2019년), GPT-3(2020년), GPT-3.5 (2022년), GPT-4(2023년), GPT-4 Turbo (2024년) 등이 출시

- 람다(LaMDA)
  - 구글이 2021년에 발표한 “획기적인” 대화 기술인 람다(대화 애플리
케이션을 위한 언어 모델)는 트랜스포머 기반 언어 모델
  - 대화를 통해 학습되며 응답의 분별력과 구체성을 대폭 개선하도록 미세 조정
  - 강점은 사람의 대화에서 흔히 발생하는 주제 표류에 대처할 수 있다는 것

- 팜(PaLM)
  - 구글 리서치가 2022년에 발표한 고밀도 디코더 전용 트랜스포머 모델
  - 매개변수 수는 5,400억 개이며 패스웨이(Pathways) 시스템을 사용해 학습
  - 고품질의 웹 문서와 서적, 위키피디아, 대화, 깃허브 코드를 포함한 여러 영어 및 다국어 데이터 집합의 조합을 사용해 학습
  - 팜-코더(PaLM-Coder)는 파이썬 전용 데이터 집합으로 미세 조정된
팜 540B 버전


- 팜-E(PaLM-E)
  - 구글이 로봇공학용으로 2023년에 구체화한 멀티모달 언어 모
델
  - 팜을 가져와서 로봇 에이전트의 센서 데이터로 보완
하여 구체화(팜-E의 ʻE’는 구체화를 의미)
  -팜-E는 팜 외에 ViT B 비전 모델도 채택했으므로 일반적인 기능의 비전 및 언어모델
  .
- 바드(Bard)
  - 구글이 2023년 출시한 람다 기반의 구글 대화형 AI 서비스
  - 2023년 3월 21일에 출시된 후 2023년 5월 10일에 일반에 공개
  - 2023년 4월에는 20개의 프로그래밍 언어로 코드를 생성하는 기능이 추가됐고, 2023년 7월에는 40가지 인간 언어 입력에 대한 지원과 함께 구글 렌즈가 통합되고 40개 이상의 언어를 사용한 텍스트-투-스피치 기능이 추가

- 라마(LLaMA)
  - 최적화된 트랜스포머 아키텍처를 사용하는 자동 회귀 언어
모델
  - 라마(대규모 언어 모델 메타 AI)는 650억 매개변수를 사용하는 “원
시” 대규모 언어 모델로, 메타 AI(전 메타-FAIR)가 2023년 2월에 출시

- 라마 2(Llama)
  - 차세대 메타 AI 대규모 언어 모델로, 2023년 1월부터 7월 사
이에 라마 1에 비해 40% 더 많은 데이터로 학습됐으며(공개적으로
사용 가능한 소스에서 2조 개의 토큰 사용), 컨텍스트 길이는 2배 더
긴 4096
  - 라마 2는 매개변수 70억 개, 130억 개, 700억 개의 여러 크기로 제공되고 사전 학습 및 미세 조정된 변형도 있음

- 클로드 2(Claude)
  - 앤트로픽(Anthropic)에서 2023년 7월에 출시된 모델로 단일 프롬프트에서 최대 10만 개의 토큰(약 7만 단어)을 수락하며, 수천 토큰의
스토리를 생성할 수 있음
  - 클로드는 구조적 데이터를 편집, 재작성, 요약, 분류, 추출할 수 있으며 내용을 기반으로 한 Q&A와 그 외의 다양한 작업이 가능